<a href="https://colab.research.google.com/github/2010626140/pIC50-calculation/blob/main/Molecular_Re_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
!apt-get update -qq
!apt-get install -y build-essential g++ swig libboost-all-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.10ubuntu1).
g++ is already the newest version (4:13.2.0-7ubuntu1).
swig is already the newest version (4.2.0-2ubuntu1).
libboost-all-dev is already the newest version (1.83.0.1ubuntu2).
0 upgraded, 0 newly installed, 0 to remove and 68 not upgraded.


In [80]:
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install --no-cache-dir vina

In [81]:
!python -m pip install meeko rdkit biopython

In [82]:
!python -m pip install --upgrade gemmi

In [83]:
!which mk_prepare_receptor.py
!which mk_prepare_ligand.py

/usr/local/bin/mk_prepare_receptor.py
/usr/local/bin/mk_prepare_ligand.py


In [84]:
!mk_prepare_receptor.py --help
!mk_prepare_ligand.py --help

usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_json JSON_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [--ignore_https_cert] [-g [GPF_FILENAME ...]]
                              [-v [VINA_BOX_FILENAME ...]]
                              [--debug_fn DEBUG_FN] [-n SET_TEMPLATE]
                              [-d DELETE_RESIDUES] [-b BLUNT_ENDS]
                              [--add_templates ADD_TEMPLATES]
                              [--cache_templates [CACHE_TEMPLATES]]
                              [--config_file CONFIG_FILE]
                              [--config_preset CONFIG_PRESET] [-x]
                              [--delete_bad_res_from_box_radius DELETE_BAD_RES_F

In [85]:
import os

print("6THA.pdb exists:", os.path.exists("/content/6THA.pdb"))

6THA.pdb exists: True


In [86]:
from Bio.PDB import PDBParser

pdb_file = "/content/6THA.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("6THA", pdb_file)

print("CHAINS AND RESIDUE COUNTS")

for chain in structure[0]:
    residues = list(chain.get_residues())
    print(
        "Chain:", chain.id,
        "| Total residues:", len(residues)
    )

print("\nHETERO RESIDUES")

for chain in structure[0]:
    for residue in chain:
        if residue.id[0] != " ":
            print(
                "Chain:", chain.id,
                "| Residue:", residue.resname,
                "| Number:", residue.id[1]
            )

CHAINS AND RESIDUE COUNTS
Chain: A | Total residues: 465

HETERO RESIDUES
Chain: A | Residue: BNG | Number: 501
Chain: A | Residue: BNG | Number: 502
Chain: A | Residue: P33 | Number: 503
Chain: A | Residue: CL | Number: 504
Chain: A | Residue: HOH | Number: 601
Chain: A | Residue: HOH | Number: 602
Chain: A | Residue: HOH | Number: 603
Chain: A | Residue: HOH | Number: 604
Chain: A | Residue: HOH | Number: 605
Chain: A | Residue: HOH | Number: 606
Chain: A | Residue: HOH | Number: 607
Chain: A | Residue: HOH | Number: 608
Chain: A | Residue: HOH | Number: 609
Chain: A | Residue: HOH | Number: 610
Chain: A | Residue: HOH | Number: 611
Chain: A | Residue: HOH | Number: 612
Chain: A | Residue: HOH | Number: 613


In [87]:
protein_lines = []
ligand_lines = []

with open(pdb_file, "r") as f:
    for line in f:

        if line.startswith("ATOM"):
            chain = line[21]
            altloc = line[16]

            if chain == "A" and altloc in (" ", "A"):
                protein_lines.append(line)

        elif line.startswith("HETATM"):
            chain = line[21]
            residue_name = line[17:20].strip()
            residue_number = int(line[22:26])

            if (
                chain == "A"
                and residue_name == "BNG"
                and residue_number == 501
            ):
                ligand_lines.append(line)

with open("/content/6THA_protein_only.pdb", "w") as f:
    f.writelines(protein_lines)
    f.write("TER\nEND\n")

with open("/content/native_ligand_BNG501.pdb", "w") as f:
    f.writelines(ligand_lines)
    f.write("END\n")

print("Protein ATOM records:", len(protein_lines))
print("Native ligand atoms:", len(ligand_lines))

Protein ATOM records: 3478
Native ligand atoms: 21


In [88]:
assert len(protein_lines) > 0, "Protein extraction failed!"
assert len(ligand_lines) == 21, "Check native ligand extraction!"

print("Protein extraction: PASS")
print("Native ligand extraction: PASS")

Protein extraction: PASS
Native ligand extraction: PASS


In [89]:
!obabel -ipdb /content/native_ligand_BNG501.pdb \
        -osdf -O /content/native_ligand.sdf -h

1 molecule converted


In [90]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

supplier = Chem.SDMolSupplier(
    "/content/native_ligand.sdf",
    removeHs=False
)

mol = supplier[0]

if mol is None:
    raise ValueError("Ligand SDF could not be read!")

print("Molecular formula:",
      rdMolDescriptors.CalcMolFormula(mol))

print("Heavy atoms:", mol.GetNumHeavyAtoms())
print("Total atoms:", mol.GetNumAtoms())
print("Bonds:", mol.GetNumBonds())

print("Canonical SMILES:",
      Chem.MolToSmiles(Chem.RemoveHs(mol)))

Molecular formula: C15H30O6
Heavy atoms: 21
Total atoms: 51
Bonds: 51
Canonical SMILES: CCCCCCCCCO[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O


In [91]:
!mk_prepare_ligand.py \
    -i /content/native_ligand.sdf \
    -o /content/native_ligand.pdbqt \
    --add_index_map

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


In [92]:
!mk_prepare_receptor.py \
    --read_pdb /content/6THA_protein_only.pdb \
    -o /content/receptor_clean \
    -p /content/receptor_clean.pdbqt

Template padding will be used.
 gasteiger charges will be read from template file 
 

Files written:
/content/receptor_clean.pdbqt <-- static (i.e., rigid) receptor input file


In [93]:
receptor_file = "/content/receptor_clean.pdbqt"

assert os.path.exists(receptor_file), "Receptor preparation failed!"

with open(receptor_file) as f:
    receptor_lines = f.readlines()

receptor_atoms = [
    line for line in receptor_lines
    if line.startswith(("ATOM", "HETATM"))
]

residues = set(
    line[17:20].strip()
    for line in receptor_atoms
)

print("Receptor exists:", os.path.exists(receptor_file))
print("Total atom records:", len(receptor_atoms))
print("Residues:", sorted(residues))
print("BNG present:", "BNG" in residues)
print("P33 present:", "P33" in residues)

Receptor exists: True
Total atom records: 4183
Residues: ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
BNG present: False
P33 present: False


In [94]:
import os

files_to_check = [
    "/content/receptor_clean.pdbqt",
    "/content/native_ligand.pdbqt"
]

for path in files_to_check:
    print("\nFile:", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Size:", round(os.path.getsize(path) / 1024, 2), "KB")


File: /content/receptor_clean.pdbqt
Exists: True
Size: 326.8 KB

File: /content/native_ligand.pdbqt
Exists: True
Size: 2.85 KB


In [95]:
receptor_file = "/content/receptor_clean.pdbqt"

with open(receptor_file) as f:
    receptor_lines = f.readlines()

receptor_atoms = [
    line for line in receptor_lines
    if line.startswith(("ATOM", "HETATM"))
]

residues = set(
    line[17:20].strip()
    for line in receptor_atoms
)

print("Receptor atom records:", len(receptor_atoms))
print("Residues:", sorted(residues))

print("BNG present:", "BNG" in residues)
print("P33 present:", "P33" in residues)

Receptor atom records: 4183
Residues: ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
BNG present: False
P33 present: False


In [96]:
ligand_file = "/content/native_ligand.pdbqt"

with open(ligand_file) as f:
    ligand_lines = f.readlines()

ligand_atoms = [
    line for line in ligand_lines
    if line.startswith(("ATOM", "HETATM"))
]

heavy_atoms = [
    line for line in ligand_atoms
    if line.split()[-1].upper() not in ("H", "HD")
]

print("Ligand atom records:", len(ligand_atoms))
print("Ligand heavy atoms:", len(heavy_atoms))

print("\nIndex mapping information:")

for line in ligand_lines:
    if "INDEX_MAP" in line:
        print(line.strip())

Ligand atom records: 25
Ligand heavy atoms: 21

Index mapping information:


In [97]:
import numpy as np

coords = []

with open("/content/native_ligand_BNG501.pdb") as f:
    for line in f:
        if line.startswith("HETATM"):
            coords.append([
                float(line[30:38]),
                float(line[38:46]),
                float(line[46:54])
            ])

coords = np.array(coords)

box_center = coords.mean(axis=0).tolist()
box_size = [22.0, 22.0, 22.0]

print("Docking box center:", box_center)
print("Docking box size:", box_size)

Docking box center: [22.076285714285717, 52.3832380952381, 14.072190476190476]
Docking box size: [22.0, 22.0, 22.0]


In [98]:
ligand_file = "/content/native_ligand.pdbqt"

with open(ligand_file, "r") as f:
    ligand_lines = f.readlines()

print("INDEX MAP REMARKS:")
for line in ligand_lines:
    if "INDEX_MAP" in line:
        print(line.strip())

print("\nFIRST 20 LINES OF LIGAND PDBQT:")
for line in ligand_lines[:20]:
    print(line.rstrip())

INDEX MAP REMARKS:

FIRST 20 LINES OF LIGAND PDBQT:
REMARK SMILES CCCCCCCCCO[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O
REMARK SMILES IDX 8 1 9 2 10 3 11 4 12 5 20 6 13 7 18 8 16 9 21 10 14 12 15 13
REMARK SMILES IDX 19 15 17 17 7 19 6 20 5 21 4 22 3 23 2 24 1 25
REMARK H PARENT 21 11 15 14 19 16 17 18
REMARK INDEX MAP 8 1 7 2 16 3 1 4 20 5 2 6 5 7 3 8 4 9 17 10 48 11 6 12 21 13
REMARK INDEX MAP 51 14 18 15 49 16 19 17 50 18 9 19 10 20 11 21 12 22 13 23
REMARK INDEX MAP 14 24 15 25
ROOT
ATOM      1  C   UNL     1      22.525  53.592  15.465  1.00  0.00    +0.029 C
ENDROOT
BRANCH   1   2
ATOM      2  C   UNL     1      22.643  52.120  15.128  1.00  0.00    +0.163 C
BRANCH   2   3
ATOM      3  O   UNL     1      23.341  51.984  13.888  1.00  0.00    -0.350 OA
BRANCH   3   4
ATOM      4  C   UNL     1      22.857  50.958  13.072  1.00  0.00    +0.281 C
ATOM      5  O   UNL     1      23.273  49.713  13.596  1.00  0.00    -0.341 OA
ATOM      6  C   UNL     1      23.427  51.149  11.671  1.00

In [100]:
from vina import Vina

# Initialize AutoDock Vina
v = Vina(
    sf_name="vina",
    cpu=2,
    seed=12345
)

# Load clean receptor
v.set_receptor(
    "/content/receptor_clean.pdbqt"
)

# Load native ligand as the docking ligand
v.set_ligand_from_file(
    "/content/native_ligand.pdbqt"
)

# Calculate maps around the crystallographic ligand
v.compute_vina_maps(
    center=[22.0762857143, 52.3832380952, 14.0721904762],
    box_size=[22.0, 22.0, 22.0]
)

# Perform redocking
v.dock(
    exhaustiveness=16,
    n_poses=20,
    min_rmsd=1.0
)

# Save all generated poses
v.write_poses(
    "/content/redocking_clean_poses.pdbqt",
    n_poses=20,
    energy_range=3
)

print("Redocking completed successfully!")

Redocking completed successfully!


In [101]:
import re
import os

pose_file = "/content/redocking_clean_poses.pdbqt"

assert os.path.exists(pose_file), "Docking output file not found!"

with open(pose_file, "r") as f:
    lines = f.readlines()

# Extract Vina scores
scores = []

for line in lines:
    if line.startswith("REMARK VINA RESULT"):
        values = re.findall(r"-?\d+\.\d+", line)
        if values:
            scores.append(float(values[0]))

# Count MODEL records
models = [line for line in lines if line.startswith("MODEL")]

print("Total poses written:", len(models))
print("Total Vina scores:", len(scores))

print("\nPOSE-WISE BINDING AFFINITIES")
for i, score in enumerate(scores, start=1):
    print(f"Pose {i}: {score:.3f} kcal/mol")

if scores:
    print("\nBest docking score:", min(scores), "kcal/mol")

Total poses written: 20
Total Vina scores: 20

POSE-WISE BINDING AFFINITIES
Pose 1: -6.802 kcal/mol
Pose 2: -6.797 kcal/mol
Pose 3: -6.761 kcal/mol
Pose 4: -6.655 kcal/mol
Pose 5: -6.621 kcal/mol
Pose 6: -6.540 kcal/mol
Pose 7: -6.504 kcal/mol
Pose 8: -6.463 kcal/mol
Pose 9: -6.444 kcal/mol
Pose 10: -6.424 kcal/mol
Pose 11: -6.398 kcal/mol
Pose 12: -6.345 kcal/mol
Pose 13: -6.339 kcal/mol
Pose 14: -6.325 kcal/mol
Pose 15: -6.241 kcal/mol
Pose 16: -6.201 kcal/mol
Pose 17: -6.184 kcal/mol
Pose 18: -6.151 kcal/mol
Pose 19: -6.109 kcal/mol
Pose 20: -6.002 kcal/mol

Best docking score: -6.802 kcal/mol


In [102]:
import numpy as np

pose_file = "/content/redocking_clean_poses.pdbqt"

poses = []
current_pose = []

with open(pose_file, "r") as f:
    for line in f:
        if line.startswith("MODEL"):
            current_pose = []

        elif line.startswith("ATOM") or line.startswith("HETATM"):
            current_pose.append([
                float(line[30:38]),
                float(line[38:46]),
                float(line[46:54])
            ])

        elif line.startswith("ENDMDL"):
            if current_pose:
                poses.append(np.array(current_pose))

native_center = np.array([
    22.0762857143,
    52.3832380952,
    14.0721904762
])

print("Pose | Atom count | Centroid distance from native ligand center")
print("-" * 65)

for i, pose in enumerate(poses, start=1):
    centroid = pose.mean(axis=0)
    distance = np.linalg.norm(centroid - native_center)

    print(
        f"{i:4d} | {len(pose):10d} | {distance:10.3f} Å"
    )

Pose | Atom count | Centroid distance from native ligand center
-----------------------------------------------------------------
   1 |         25 |      1.848 Å
   2 |         25 |      1.573 Å
   3 |         25 |      0.969 Å
   4 |         25 |      2.504 Å
   5 |         25 |      1.745 Å
   6 |         25 |      2.517 Å
   7 |         25 |      2.048 Å
   8 |         25 |      2.430 Å
   9 |         25 |      3.134 Å
  10 |         25 |      2.879 Å
  11 |         25 |      4.716 Å
  12 |         25 |      2.608 Å
  13 |         25 |      2.505 Å
  14 |         25 |      1.806 Å
  15 |         25 |      1.786 Å
  16 |         25 |      2.203 Å
  17 |         25 |      1.147 Å
  18 |         25 |      1.988 Å
  19 |         25 |      2.166 Å
  20 |         25 |      1.172 Å


In [103]:
import numpy as np
from rdkit import Chem

# File paths
native_pdb = "/content/native_ligand_BNG501.pdb"
ligand_sdf = "/content/native_ligand.sdf"
ligand_pdbqt = "/content/native_ligand.pdbqt"
poses_file = "/content/redocking_clean_poses.pdbqt"

# --------------------------------------------------
# 1. Read native ligand coordinates from original PDB
# --------------------------------------------------

native_atoms = []

with open(native_pdb) as f:
    for line in f:
        if line.startswith("HETATM"):
            element = line[76:78].strip().upper()

            if element != "H":
                native_atoms.append({
                    "atom_name": line[12:16].strip(),
                    "element": element,
                    "coord": np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                })

native_coords = np.array([a["coord"] for a in native_atoms])

# --------------------------------------------------
# 2. Read the prepared ligand SDF
# --------------------------------------------------

mol = Chem.SDMolSupplier(
    ligand_sdf,
    removeHs=False
)[0]

if mol is None:
    raise ValueError("Could not read ligand SDF.")

conf = mol.GetConformer()

sdf_heavy_indices = [
    atom.GetIdx()
    for atom in mol.GetAtoms()
    if atom.GetSymbol() != "H"
]

sdf_heavy_coords = np.array([
    list(conf.GetAtomPosition(i))
    for i in sdf_heavy_indices
])

print("Native PDB heavy atoms:", len(native_coords))
print("SDF heavy atoms:", len(sdf_heavy_coords))

# --------------------------------------------------
# 3. Validate atom order using coordinates
# --------------------------------------------------

if len(native_coords) != len(sdf_heavy_coords):
    raise ValueError("Heavy atom counts do not match.")

coordinate_differences = np.linalg.norm(
    native_coords - sdf_heavy_coords,
    axis=1
)

print(
    "Maximum PDB-SDF coordinate difference:",
    coordinate_differences.max(),
    "Å"
)

if not np.allclose(
    native_coords,
    sdf_heavy_coords,
    atol=0.002
):
    raise ValueError(
        "PDB and SDF atom order/coordinates do not match. "
        "Stop before calculating RMSD."
    )

print("PDB-SDF atom order validation: PASS")

# --------------------------------------------------
# 4. Read Meeko INDEX MAP
# --------------------------------------------------

index_map = {}

with open(ligand_pdbqt) as f:
    for line in f:
        if line.startswith("REMARK INDEX MAP"):
            values = list(map(int, line.split()[3:]))

            for i in range(0, len(values), 2):
                original_idx = values[i]
                pdbqt_serial = values[i + 1]

                index_map[original_idx] = pdbqt_serial

# Map original SDF heavy atom indices (1-based)
# to PDBQT atom serial numbers.

heavy_mapping = {}

for sdf_idx in sdf_heavy_indices:
    original_idx = sdf_idx + 1

    if original_idx not in index_map:
        raise ValueError(
            f"Missing INDEX MAP entry for atom {original_idx}"
        )

    heavy_mapping[original_idx] = index_map[original_idx]

print("Mapped heavy atoms:", len(heavy_mapping))

if len(heavy_mapping) != 21:
    raise ValueError("Expected 21 mapped heavy atoms.")

# --------------------------------------------------
# 5. Extract all docked poses by atom serial number
# --------------------------------------------------

poses = []
current_pose = None

with open(poses_file) as f:
    for line in f:

        if line.startswith("MODEL"):
            current_pose = {}

        elif line.startswith(("ATOM", "HETATM")) and current_pose is not None:
            serial = int(line[6:11])

            current_pose[serial] = np.array([
                float(line[30:38]),
                float(line[38:46]),
                float(line[46:54])
            ])

        elif line.startswith("ENDMDL"):
            if current_pose is not None:
                poses.append(current_pose)
                current_pose = None

# --------------------------------------------------
# 6. Calculate unaligned heavy-atom RMSD
# --------------------------------------------------

results = []

for pose_number, pose in enumerate(poses, start=1):

    native_matched = []
    docked_matched = []

    for original_idx, pdbqt_serial in heavy_mapping.items():

        if pdbqt_serial not in pose:
            raise ValueError(
                f"Pose {pose_number}: missing atom serial {pdbqt_serial}"
            )

        native_matched.append(native_coords[original_idx - 1])
        docked_matched.append(pose[pdbqt_serial])

    native_matched = np.array(native_matched)
    docked_matched = np.array(docked_matched)

    rmsd = np.sqrt(
        np.mean(
            np.sum(
                (native_matched - docked_matched) ** 2,
                axis=1
            )
        )
    )

    results.append((pose_number, rmsd))

# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\nUNALIGNED HEAVY-ATOM REDOCKING RMSD")
print("-----------------------------------")

for pose_number, rmsd in results:
    print(f"Pose {pose_number:2d}: {rmsd:.3f} Å")

best_rmsd_pose = min(results, key=lambda x: x[1])

print("\nLowest RMSD:")
print("Pose:", best_rmsd_pose[0])
print("RMSD:", round(best_rmsd_pose[1], 3), "Å")

Native PDB heavy atoms: 21
SDF heavy atoms: 21
Maximum PDB-SDF coordinate difference: 0.0 Å
PDB-SDF atom order validation: PASS
Mapped heavy atoms: 21

UNALIGNED HEAVY-ATOM REDOCKING RMSD
-----------------------------------
Pose  1: 4.225 Å
Pose  2: 7.349 Å
Pose  3: 7.682 Å
Pose  4: 7.526 Å
Pose  5: 4.774 Å
Pose  6: 4.581 Å
Pose  7: 5.770 Å
Pose  8: 5.316 Å
Pose  9: 6.688 Å
Pose 10: 5.720 Å
Pose 11: 6.651 Å
Pose 12: 5.640 Å
Pose 13: 4.593 Å
Pose 14: 6.800 Å
Pose 15: 4.328 Å
Pose 16: 4.608 Å
Pose 17: 3.311 Å
Pose 18: 2.926 Å
Pose 19: 3.870 Å
Pose 20: 3.302 Å

Lowest RMSD:
Pose: 18
RMSD: 2.926 Å
